# EXP002 Qwen3.5-2B QLoRA — stage 00

Continue from optimizer step 0 through 250.


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
REPO_REV = '654b284'
REPO_ROOT = Path('/kaggle/working/spider')
subprocess.run(['git', 'clone', 'https://github.com/yogesh-dhande/spider.git', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))


In [ ]:
%pip install -q --progress-bar off -r requirements/experiment2-kaggle.txt


In [ ]:
from spider.workflow import find_prepared_data
prepared = find_prepared_data('/kaggle/input')
os.environ['SPIDER_DATA_DIR'] = str(prepared)
print({'event': 'prepared_data_mounted', 'path': str(prepared)})


In [ ]:
from spider.workflow import find_completed_training_outputs
assert not find_completed_training_outputs('/kaggle/input')
print({'event': 'training_from_base_model', 'completed_step': 0})


In [ ]:
from spider.workflow import gpu_summary
print({'event': 'gpu_inventory', **gpu_summary()})


In [ ]:
from spider.ddp_smoke import torchrun_command
command = torchrun_command(
    'configs/experiment2.yaml', 250, 2,
    8, resume='auto'
)
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join(
    value for value in (str(REPO_ROOT / 'src'), env.get('PYTHONPATH')) if value
)
print({'event': 'distributed_training_start', 'command': command})
subprocess.run(command, check=True, env=env)
adapter = REPO_ROOT / 'outputs/experiment2/adapter/final'
state = json.loads(
    (REPO_ROOT / 'outputs/experiment2/training_state.json').read_text()
)
assert state['start_step'] == 0, state
assert state['completed_step'] == 250, state
assert state['world_size'] == 2, state
assert state['gradient_accumulation_steps'] == 8, state
assert state['effective_batch_size'] == 16, state
print({'event': 'stage_runner_complete', 'state': state, 'adapter': str(adapter)})
